# IndicTrans2 Fine-tuning: English → Odia (with Odia Script Fix)
**Fixes:**
- Pinned compatible versions (`transformers>=4.51`, `torch>=2.5`, `numpy>=2.1`)
- Fallback import for `IndicProcessor` (handles Cython build issues)
- `forced_bos_token_id=ory_Orya` to prevent Devanagari output
- Script verification after every translation batch

> ⚠️ **Run Step 0 first, then use Runtime → Restart and run all (or restart kernel before Step 2)**

## Step 0: Install Dependencies
**After this cell finishes → RESTART THE RUNTIME, then continue from Step 1**

In [3]:
# Pin compatible versions — critical for IndicTransToolkit to work
!pip install -q "numpy>=2.1" "torch>=2.5" "transformers>=4.51"
!pip install -q sacrebleu sentencepiece peft accelerate

# Install IndicTransToolkit (provides IndicProcessor for preprocessing)
!pip install -q indictranstoolkit

print('\n✅ Done. NOW RESTART THE RUNTIME before continuing!')
print('   Runtime → Restart session  (Colab)  or  Kernel → Restart (Jupyter)')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 111.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.2 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.4.2 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 546.3/546.3 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 7.4 MB/s eta 0:00:00

✅ Done. NOW RESTART THE RUNTIME before continuing!
   Runtime → Restart session  (

## Step 1: Load Dataset

In [1]:
import pandas as pd

# ── Mount Google Drive if using Colab ──
# from google.colab import drive
# drive.mount('/content/drive')
# DATA_PATH = '/content/drive/MyDrive/train.final'

DATA_PATH = 'train.final'  # <-- change to your actual path

df = pd.read_csv(DATA_PATH, sep='\t', header=None, names=['source', 'english', 'odia'])
df = df.dropna(subset=['english', 'odia']).reset_index(drop=True)
print(f'Total rows: {len(df)}')
print(df.head(2))

Total rows: 23789
    source                                            english  \
0  genesis  In the beginning God created the heaven and th...   
1  genesis  And the earth was without form, and void; and ...   

                                                odia  
0       ଆରମ୍ଭରେ ପରମେଶ୍ବର ଆକାଶ ଓ ପୃଥିବୀକୁ ସୃଷ୍ଟି କଲେ।  
1  ପୃଥିବୀ ସେତବେେଳେ ସଂପୂରନ୍ଭାବେ ଶୂନ୍ଯ ଓ କିଛି ନଥିଲା...  


In [2]:
# Verify Odia script (Odia = U+0B00–U+0B7F, Devanagari = U+0900–U+097F)
def script_of(text):
    for ch in text:
        if '\u0B00' <= ch <= '\u0B7F': return 'Odia'
        if '\u0900' <= ch <= '\u097F': return 'Devanagari'
    return 'Other'

scripts = df['odia'].apply(script_of)
print('Script distribution in your dataset:')
print(scripts.value_counts())
print('\nExpected: mostly Odia. If Devanagari — fix your dataset first!')

Script distribution in your dataset:
odia
Odia          23786
Other             2
Devanagari        1
Name: count, dtype: int64

Expected: mostly Odia. If Devanagari — fix your dataset first!


In [3]:
# Split: 10000 train, 300 eval
TRAIN_SIZE = 10000
EVAL_SIZE  = 300

df = df.sample(frac=1, random_state=42).reset_index(drop=True)
train_df = df.iloc[:TRAIN_SIZE].reset_index(drop=True)
eval_df  = df.iloc[TRAIN_SIZE:TRAIN_SIZE + EVAL_SIZE].reset_index(drop=True)

print(f'Train: {len(train_df)}, Eval: {len(eval_df)}')

Train: 10000, Eval: 300


## Step 2: Load Model + IndicProcessor

**Root cause of Devanagari bug:** IndicTrans2 internally uses script tags. If `IndicProcessor` is not used for preprocessing, or if `forced_bos_token_id` is wrong/missing, the decoder produces Devanagari instead of Odia. The fix is:
1. Always preprocess with `IndicProcessor`
2. Always pass `forced_bos_token_id = tokenizer.convert_tokens_to_ids('ory_Orya')`

In [4]:
!pip install "numpy<2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 34.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.2
    Uninstalling numpy-2.4.2:
      Successfully uninstalled numpy-2.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_vers

In [4]:
# ── IndicProcessor import with fallback ──
try:
    from IndicTransToolkit import IndicProcessor
    print('Imported IndicProcessor from IndicTransToolkit')
except ImportError:
    # Fallback for Cython build issues
    from IndicTransToolkit.IndicTransToolkit import IndicProcessor
    print('Imported IndicProcessor via fallback path')

import torch
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = 'ai4bharat/indictrans2-en-indic-1B'
SRC_LANG   = 'eng_Latn'
TGT_LANG   = 'ory_Orya'   # ← Odia script (NOT ory_Deva)

print('\nLoading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# ── Verify ory_Orya token exists ──
ory_orya_id = tokenizer.convert_tokens_to_ids(TGT_LANG)
print(f'Token "{TGT_LANG}" → ID: {ory_orya_id}')
assert ory_orya_id != tokenizer.unk_token_id, \
    f'ERROR: ory_Orya mapped to UNK! Check tokenizer. UNK id = {tokenizer.unk_token_id}'
print('✅ ory_Orya token confirmed — Odia script output will be correct.')

Imported IndicProcessor from IndicTransToolkit

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

tokenization_indictrans.py:   0%|          | 0.00/8.04k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-1B:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

dict.TGT.json:   0%|          | 0.00/3.39M [00:00<?, ?B/s]

model.SRC:   0%|          | 0.00/759k [00:00<?, ?B/s]

model.TGT:   0%|          | 0.00/3.26M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

Token "ory_Orya" → ID: 69
✅ ory_Orya token confirmed — Odia script output will be correct.


In [1]:
!pip install "transformers==4.38.2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 29.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 33.6 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.1
    Uninstalling huggingface_hub-1.4.1:
      Successfully uninstalled huggingface_hub-1.4.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the follow

In [5]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

print('Loading base model...')
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16 if device == 'cuda' else torch.float32,
).to(device)
base_model.eval()

ip = IndicProcessor(inference=True)
print('✅ Model loaded.')

Using device: cuda
Loading base model...


config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

configuration_indictrans.py:   0%|          | 0.00/14.2k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-1B:
- configuration_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


modeling_indictrans.py:   0%|          | 0.00/79.8k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-1B:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/4.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

✅ Model loaded.


In [ ]:
# ── Quick translation test ──
test_sentences = [
    "God created the heaven and the earth.",
    "The light was good.",
    "Write your own sentence here."
]

preprocessed = ip.preprocess_batch(test_sentences, src_lang=SRC_LANG, tgt_lang=TGT_LANG)
inputs = tokenizer(preprocessed, truncation=True, padding='longest',
                   max_length=256, return_tensors='pt').to(device)

with torch.no_grad():
    outputs = base_model.generate(
        **inputs,
        num_beams=4,
        max_length=256,
        forced_bos_token_id=ory_orya_id,
    )

translated = tokenizer.batch_decode(outputs, skip_special_tokens=True,
                                    clean_up_tokenization_spaces=True)
translated = ip.postprocess_batch(translated, lang=TGT_LANG)

print('Results:\n')
for eng, odi in zip(test_sentences, translated):
    print(f'EN  : {eng}')
    print(f'OD  : {odi}  [{script_of(odi)}]')
    print()

Results:

EN  : God created the heaven and the earth.
OD  : ମରିଯୁ ଭଗବାନ ଆକାଶମଣ୍ଡଳ ଓ ପୃଥିବୀ ସୃଷ୍ଟି କରିଛନ୍ତି।  [Odia]

EN  : The light was good.
OD  : ମରିଯୁ ଆଲୋକ ଭଲ ଥିଲା।  [Odia]

EN  : Write your own sentence here.
OD  : ମରିଯୁ ଏଠାରେ ନିଜର ବାକ୍ଯ଼ ଲେଖନ୍ତୁ।  [Odia]



## Step 3: Baseline BLEU (Before Fine-tuning)

In [6]:
from sacrebleu.metrics import BLEU, CHRF

def translate_batch(sentences, model, batch_size=8):
    """Translate English → Odia using IndicProcessor (prevents Devanagari output)"""
    model.eval()
    all_out = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i : i + batch_size]

        # IndicProcessor adds script markers — critical for correct script output
        preprocessed = ip.preprocess_batch(batch, src_lang=SRC_LANG, tgt_lang=TGT_LANG)

        inputs = tokenizer(
            preprocessed, truncation=True, padding='longest',
            max_length=256, return_tensors='pt'
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                num_beams=4,
                max_length=256,
                forced_bos_token_id=ory_orya_id,  # ← THE KEY FIX for script
            )

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True,
                                         clean_up_tokenization_spaces=True)
        decoded = ip.postprocess_batch(decoded, lang=TGT_LANG)
        all_out.extend(decoded)

    # Warn if any non-Odia output slips through
    for t in all_out:
        s = script_of(t)
        if s == 'Devanagari':
            print(f'⚠️  Still got Devanagari output: {t[:50]}')
    return all_out


def score(hyps, refs):
    bleu = BLEU(tokenize='char').corpus_score(hyps, [refs])
    chrf = CHRF(word_order=2).corpus_score(hyps, [refs])   # chrF++
    return bleu, chrf


# Use 100 examples for speed (remove [:100] for full eval set)
eval_src = eval_df['english'].tolist()[:100]
eval_ref = eval_df['odia'].tolist()[:100]

print('Evaluating baseline model...')
baseline_hyp = translate_batch(eval_src, base_model)

baseline_bleu, baseline_chrf = score(baseline_hyp, eval_ref)
print(f'\n📊 BASELINE  →  BLEU: {baseline_bleu.score:.2f}   chrF++: {baseline_chrf.score:.2f}')

print('\nSample (script check):')
for eng, ref, hyp in zip(eval_src[:3], eval_ref[:3], baseline_hyp[:3]):
    print(f'  EN : {eng[:70]}')
    print(f'  REF: {ref[:60]}  [{script_of(ref)}]')
    print(f'  OUT: {hyp[:60]}  [{script_of(hyp)}]')
    print()

Evaluating baseline model...

📊 BASELINE  →  BLEU: 30.08   chrF++: 23.52

Sample (script check):
  EN : And he said unto them, I must preach the kingdom of God to other citie
  REF: କିନ୍ତୁ ଯୀଶୁ ସମାନଙ୍କେୁ କହିଲେ, "ପରମେଶ୍ବରଙ୍କ ରାଜ୍ଯର ସୁସମାଚାର ମା  [Odia]
  OUT: ମରିଯୁ ସେ ସେମାନଙ୍କୁ କହିଲେ, ମୋତେ ଅନ୍ଯ଼ ନଗରମାନଙ୍କରେ ମଧ୍ଯ଼ ପରମେଶ  [Odia]

  EN : And Mizraim begat Ludim, and Anamim, and Lehabim, and Naphtuhim
  REF: ମିଶ୍ରଯୀମ (ମିଶର), ଲୂଦୀଯ, ଅନାମୀଯ, ଲହାବୀଯ ଓ ନପ୍ତୂହୀଯମାନଙ୍କର ପିତ  [Odia]
  OUT: ମରିଯୁ ମିଜ୍ରାଇମ୍ ଲୁଦିମ୍, ଅନାମିମ୍, ଲହାବିମ୍, ଏବଂ ନପ୍ତୁହିମ୍ଙ୍କୁ   [Odia]

  EN : They reap every one his corn in the field: and they gather the vintage
  REF: ଗରିବ ଲୋକମାନେ ବହୁତ ବିଳମ୍ବିତ ରାତ୍ରିୟାଏ ଫସଲ ଅମଳ କରନ୍ତି। ସମେନେ   [Odia]
  OUT: ମରିଯୁ ସେମାନେ ସମସ୍ତେ କ୍ଷେତରେ ନିଜ ନିଜ ଶସ୍ଯ଼ ଅମଳ କରନ୍ତି। ଏବଂ ସେ  [Odia]



## Step 4: Fine-tune with LoRA

In [2]:
!pip install "peft==0.10.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 6.2 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.18.1
    Uninstalling peft-0.18.1:
      Successfully uninstalled peft-0.18.1


In [7]:
from torch.utils.data import Dataset
from peft import get_peft_model, LoraConfig, TaskType

# ── Dataset ──
class OdiaDataset(Dataset):
    def __init__(self, src_list, tgt_list, max_len=256):
        self.src = src_list
        self.tgt = tgt_list
        self.ip_train = IndicProcessor(inference=False)
        self.max_len = max_len

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        src_text = self.ip_train.preprocess_batch(
            [self.src[idx]], src_lang=SRC_LANG, tgt_lang=TGT_LANG)[0]
        tgt_text = self.ip_train.preprocess_batch(
            [self.tgt[idx]], src_lang=TGT_LANG, tgt_lang=SRC_LANG)[0]

        model_in = tokenizer(
            src_text, truncation=True, max_length=self.max_len,
            padding='max_length', return_tensors='pt'
        )
        with tokenizer.as_target_tokenizer():
            label_enc = tokenizer(
                tgt_text, truncation=True, max_length=self.max_len,
                padding='max_length', return_tensors='pt'
            )

        labels = label_enc['input_ids'].squeeze().clone()
        labels[labels == tokenizer.pad_token_id] = -100

        return {
            'input_ids':      model_in['input_ids'].squeeze(),
            'attention_mask': model_in['attention_mask'].squeeze(),
            'labels':         labels,
        }

train_dataset = OdiaDataset(train_df['english'].tolist(), train_df['odia'].tolist())
print(f'Train dataset: {len(train_dataset)} examples')

Train dataset: 10000 examples


In [8]:
# ── Load fresh model for training ──
ft_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME, trust_remote_code=True, torch_dtype=torch.float32
).to(device)

# ── Apply LoRA ──
lora_cfg = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=8,
    lora_alpha=16,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'out_proj', 'fc1', 'fc2'],
    lora_dropout=0.1,
    bias='none',
)
ft_model.enable_input_require_grads()
ft_model = get_peft_model(ft_model, lora_cfg)
ft_model.print_trainable_parameters()

trainable params: 8,847,360 || all params: 1,124,390,912 || trainable%: 0.7868580140213727


In [ ]:
import torch, gc

# Delete old variables if they exist
try:
    del base_model
    del ft_model
except NameError:
    pass

# Force Python garbage collection and empty CUDA cache
gc.collect()
torch.cuda.empty_cache()

print(f"Memory cleared! Currently allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

Memory cleared! Currently allocated: 14.24 GB


In [ ]:
ft_model.print_trainable_parameters()

trainable params: 8,847,360 || all params: 1,124,390,912 || trainable%: 0.7868580140213727


In [3]:
!pip install "accelerate==0.27.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.0/280.0 kB 6.9 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0


In [9]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

training_args = Seq2SeqTrainingArguments(
    output_dir='./it2_odia_lora',
    num_train_epochs=3,
    per_device_train_batch_size=2,    # lower to 2 if OOM on T4
    gradient_accumulation_steps=8,    # effective batch = 32
    learning_rate=5e-5,
    warmup_steps=100,
    logging_steps=50,
    save_steps=500,
    save_total_limit=2,
    predict_with_generate=False,
    fp16=(device == 'cuda'),
    gradient_checkpointing=False,
    report_to='none',

)

trainer = Seq2SeqTrainer(
    model=ft_model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=ft_model, pad_to_multiple_of=8),
)

print('Starting fine-tuning... (~1-3 hrs on T4)')
trainer.train()
print('✅ Fine-tuning done!')

/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:450: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Starting fine-tuning... (~1-3 hrs on T4)


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:3892: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Step,Training Loss
50,5.221000
100,4.480000
150,3.386800
200,2.803200
250,2.622700
300,2.603300
350,2.584300
400,2.579100
450,2.464000
500,2.554700


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:3892: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:3892: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input te

✅ Fine-tuning done!


In [ ]:
import torch, gc

# Delete old variables if they exist
try:
    del trainer
except NameError:
    pass

# Force Python garbage collection and empty CUDA cache
gc.collect()
torch.cuda.empty_cache()

print(f"Memory cleared! Currently allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

Memory cleared! Currently allocated: 14.24 GB


In [10]:
# Save LoRA adapter
ft_model.save_pretrained('./it2_odia_lora_adapter')
tokenizer.save_pretrained('./it2_odia_lora_adapter')
print('Saved to ./it2_odia_lora_adapter')

# Optional: Save to Google Drive
# !cp -r ./it2_odia_lora_adapter /content/drive/MyDrive/it2_odia_lora_adapter

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Saved to ./it2_odia_lora_adapter


## Step 5: Post Fine-tuning BLEU + Comparison

In [11]:
ft_model.eval()

print('Evaluating fine-tuned model...')
finetuned_hyp = translate_batch(eval_src, ft_model)

ft_bleu, ft_chrf = score(finetuned_hyp, eval_ref)

# ── Results table ──
print('\n' + '='*52)
print('       BEFORE vs AFTER FINE-TUNING')
print('='*52)
print(f'  Metric  |  Baseline  |  Fine-tuned  |  Delta')
print(f'  --------+------------+--------------+-------')
print(f'  BLEU    |  {baseline_bleu.score:6.2f}    |  {ft_bleu.score:8.2f}    |  {ft_bleu.score - baseline_bleu.score:+.2f}')
print(f'  chrF++  |  {baseline_chrf.score:6.2f}    |  {ft_chrf.score:8.2f}    |  {ft_chrf.score - baseline_chrf.score:+.2f}')
print('='*52)

Evaluating fine-tuned model...

       BEFORE vs AFTER FINE-TUNING
  Metric  |  Baseline  |  Fine-tuned  |  Delta
  --------+------------+--------------+-------
  BLEU    |   30.08    |     33.07    |  +3.00
  chrF++  |   23.52    |     26.00    |  +2.47


In [12]:
# Side-by-side sample comparison
print('Sample translations comparison:')
for i in range(5):
    print(f'[{i+1}] EN      : {eval_src[i][:75]}')
    print(f'     REF     : {eval_ref[i][:65]}  [{script_of(eval_ref[i])}]')
    print(f'     BASELINE: {baseline_hyp[i][:65]}  [{script_of(baseline_hyp[i])}]')
    print(f'     FINETUNED: {finetuned_hyp[i][:65]}  [{script_of(finetuned_hyp[i])}]')
    print()

Sample translations comparison:
[1] EN      : And he said unto them, I must preach the kingdom of God to other cities als
     REF     : କିନ୍ତୁ ଯୀଶୁ ସମାନଙ୍କେୁ କହିଲେ, "ପରମେଶ୍ବରଙ୍କ ରାଜ୍ଯର ସୁସମାଚାର ମାେତେ ଅ  [Odia]
     BASELINE: ମରିଯୁ ସେ ସେମାନଙ୍କୁ କହିଲେ, ମୋତେ ଅନ୍ଯ଼ ନଗରମାନଙ୍କରେ ମଧ୍ଯ଼ ପରମେଶ୍ୱରଙ୍  [Odia]
     FINETUNED: ମରିଯୁ ଯୀଶୁ ସମାନଙ୍କେୁ କହିଲେ, "ଆମ୍ଭେ ଅନ୍ଯ଼ ନଗର ରେ ମଧ୍ଯ ପରମେଶ୍ବରଙ୍କ   [Odia]

[2] EN      : And Mizraim begat Ludim, and Anamim, and Lehabim, and Naphtuhim
     REF     : ମିଶ୍ରଯୀମ (ମିଶର), ଲୂଦୀଯ, ଅନାମୀଯ, ଲହାବୀଯ ଓ ନପ୍ତୂହୀଯମାନଙ୍କର ପିତା ଥିଲ  [Odia]
     BASELINE: ମରିଯୁ ମିଜ୍ରାଇମ୍ ଲୁଦିମ୍, ଅନାମିମ୍, ଲହାବିମ୍, ଏବଂ ନପ୍ତୁହିମ୍ଙ୍କୁ ଜନ୍ମ   [Odia]
     FINETUNED: ମରିଯୁ ମିଜ୍ରାଯିମ ଲୁଦୀମ, ଅନାମୀମ, ଲହାବୀମ ଓ ନପ୍ତହୀମଙ୍କର ପିତା ହେଲେ।  [Odia]

[3] EN      : They reap every one his corn in the field: and they gather the vintage of t
     REF     : ଗରିବ ଲୋକମାନେ ବହୁତ ବିଳମ୍ବିତ ରାତ୍ରିୟାଏ ଫସଲ ଅମଳ କରନ୍ତି। ସମେନେ ନିଶ୍ଚ  [Odia]
     BASELINE: ମରିଯୁ ସେମାନେ ସମସ୍ତେ କ୍ଷେତରେ ନିଜ ନିଜ ଶସ୍ଯ଼ ଅମଳ କରନ୍ତି। ଏବଂ ସେମାନେ   [Odi

## How to Reload the Fine-tuned Model Later

```python
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from peft import PeftModel

base = AutoModelForSeq2SeqLM.from_pretrained('ai4bharat/indictrans2-en-indic-1B', trust_remote_code=True)
model = PeftModel.from_pretrained(base, './it2_odia_lora_adapter')
model = model.merge_and_unload()  # merges LoRA into weights for faster inference
tokenizer = AutoTokenizer.from_pretrained('./it2_odia_lora_adapter', trust_remote_code=True)
```